# Week 2, Lab 4 — Guardrails


In [1]:
WEEK = 'Week 2'
LAB = 'Lab 4 — guardrails'

import sys
from pathlib import Path

def _course_root() -> Path:
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if (p / "shared" / "course_runtime.py").exists():
            return p
    for c in [
        here / "agentic_ai_local",
        Path("/content/agentic_ai_local"),
        Path("/content"),
    ]:
        if (c / "shared" / "course_runtime.py").exists():
            return c
    return here

ROOT = _course_root()
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from shared.course_runtime import (
    detect_backend,
    print_banner,
    local_chat,
    calculator,
    lookup_fact,
    today_date,
    extract_json_object,
    parse_tool_call,
    openai_client_kwargs,
    get_langchain_llm,
    TOOL_SCHEMAS,
    MOCK_KB,
)

BACKEND = print_banner(WEEK, LAB)
print("If import failed, unzip/clone the WHOLE course folder (not a single notebook).")


Week 2 / Lab 4 — guardrails
Backend: ollama
Need Ollama running: `ollama serve` and `ollama pull llama3.2:1b`
If import failed, unzip/clone the WHOLE course folder (not a single notebook).


In [2]:
cfg = openai_client_kwargs()
print(cfg)

from openai import AsyncOpenAI
from agents import Agent, Runner, OpenAIChatCompletionsModel, function_tool, handoff

client = AsyncOpenAI(base_url=cfg["base_url"], api_key=cfg["api_key"])
model = OpenAIChatCompletionsModel(model=cfg["model"], openai_client=client)


{'base_url': 'http://localhost:11434/v1', 'api_key': 'ollama', 'model': 'qwen2.5:3b'}


In [3]:
from agents import input_guardrail, GuardrailFunctionOutput, InputGuardrailTripwireTriggered

BLOCKLIST = ("password", "api key", "api_key", "credit card")

@input_guardrail
async def no_secrets(ctx, agent, input_data):
    text = input_data if isinstance(input_data, str) else str(input_data)
    tripped = any(w in text.lower() for w in BLOCKLIST)
    return GuardrailFunctionOutput(
        output_info={"text": text, "tripped": tripped},
        tripwire_triggered=tripped,
    )

agent = Agent(
    name="GuardedTutor",
    instructions="Help with the agentic AI course. Be brief.",
    model=model,
    input_guardrails=[no_secrets],
)

async def try_run(prompt: str):
    try:
        result = await Runner.run(agent, prompt)
        print("OK:", result.final_output)
    except InputGuardrailTripwireTriggered:
        print("BLOCKED by input guardrail:", prompt)

await try_run("What is an agent?")
await try_run("Here is my api key sk-test — store it.")


OPENAI_API_KEY is not set, skipping trace export


OK: An agent can be defined as a system that operates in an environment to achieve its goals, often by making decisions and taking actions. Agents continuously perceive their environment through sensors, interpret those perceptions using its knowledge base, and take actions to change the environment via effectors. They are fundamental concepts in areas like robotics, AI planning, game playing, and many more.
BLOCKED by input guardrail: Here is my api key sk-test — store it.


OPENAI_API_KEY is not set, skipping trace export


Python guardrails are more reliable than hoping the model will refuse.\n\n**Next:** mini-project.
